# Introduction

Notebook for generating or collecting results for Figure 1.

**IMPORTANT:** Requires `01_exp_regression.ipynb` to run first.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(1)

In [ ]:
import sys
import logging
import re
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import scipy
import sklearn.cluster
import sklearn.compose

In [ ]:
import nedis.base
import nedis.visualization
import nedis.cluster.leidenalg

In [ ]:
import nedis.data.synthetic
import nedis.data.tasks
from nedis.visualization import grouped_spline_plot

In [ ]:
# %%
# computing

default_n_jobs = 20

# reading number of jobs from environment variables
import os
n_jobs = int(os.getenv('N_JOBS', default=str(default_n_jobs)))

print(f"Number of jobs: {n_jobs}")

# %%
# plotting

# init matplotlib defaults (for Nima)
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

# persistance

# tracking_uri = "http://171.65.176.47:5000"
# tracking_uri = "http://localhost:5000"

# Setup

In [ ]:
%run -m rpy2.situation

In [ ]:
logging.basicConfig(stream=sys.stdout)
logging.getLogger("nedis").setLevel(level=logging.DEBUG)

In [ ]:
output_dir = pathlib.Path("../_out/figure1")
output_dir.mkdir(parents=True, exist_ok=True)

# Load Data

In [ ]:
data_path = "../data"

In [ ]:
task_list_complete = nedis.data.tasks.load_task_list(
    data_path=data_path,
    include=["multiomics", "pree"])

In [ ]:
for i, t in enumerate(task_list_complete):
    print(i, t["name"])

In [ ]:
def select_task(task_id):
    # set task
    if isinstance(task_id, int):
        task = task_list_complete[task_id]
    else:
        task = [t for t in task_list_complete if task_id == t['id']]
        assert len(task) == 1, f"{task_id}, {[t['name'] for t in task]}"
        task = task[0]
    print(task["name"])
    return task

# Functions

In [ ]:
# test tests for mean and variance change
samples = [np.random.normal(size=100) for i in range(10)] + [np.random.normal(size=100)]
r1, p1 = scipy.stats.kruskal(*samples) # median
r2, p2 = scipy.stats.levene(*samples)  # variance
threshold = 0.5
print(p1, p2)
print(p1> threshold, p2 > threshold)

## Functions

In [ ]:
def prepare_variables(task, cache):
    
    timepoints = task["timepoints"]
    cache["timepoints_unique"] = np.unique(timepoints)
    cache["timepoints_idx"] = scipy.stats.rankdata(timepoints, method="dense") - 1
    cache["n_timepoints"] = len(cache["timepoints_unique"])

    groups = task["groups"]
    cache["groups_unique"] = np.unique(groups)
    cache["n_groups"] = len(cache["groups_unique"])

    entities = task["entities"]
    cache["entities_unique"] = np.unique(entities)
    cache["n_entities"] = len(cache["entities_unique"])
    
    print(f"  * Task statistics:")
    print(f"    * timepoints: {cache['n_timepoints']}")
    print(f"    * groups:     {cache['n_groups']}") 
    print(f"    * entities:   {cache['n_entities']}")

In [ ]:
def check_homoescedasticity(task, cache, plot_pvalue_threshold=0.9, plot_n_features=3):
    
    print("  * Find homoscedastic variables")
    
    # shortcut for features
    X = task["features"]
    
    # calculate p-values for mean and variance change
    
    pvalues = np.zeros((2, X.shape[1]))
    for i in range(X.shape[1]):
        samples = [X[t == task["timepoints"], i] for t in cache["timepoints_unique"]]
        r1, p1 = scipy.stats.kruskal(*samples)
        r2, p2 = scipy.stats.levene(*samples)
        pvalues[:,i] = (p1, p2)
    cache["pvalues_mean_var"] = pvalues
        
    # plot thresholds
    
    pvalue_thresholds = np.arange(0, 1.05, 0.05)
    pvalue_counts = np.array([np.all(pvalues > t, axis=0).sum() for t in pvalue_thresholds])

    fig, ax = plt.subplots()
    ax.plot(pvalue_thresholds, pvalue_counts)
    for i, t in enumerate(pvalue_thresholds):
        ax.axvline(t, color="grey", linestyle="--")
        ax.annotate(pvalue_counts[i], (t,pvalue_counts[i]), ha="center")
    
    ax.axvline(0.05, color="red", linestyle="--", linewidth=5)
    ax.set(xlabel="pvalue threshold (pvalue > x)", ylabel="number of heteroscedastic features", title="")

    # plot most non-changing features
    
    selected = np.argwhere(np.all(pvalues > plot_pvalue_threshold, axis=0)).flatten()

    for s1 in selected[:plot_n_features]:

        fix, ax = plt.subplots()

        # TODO: does no work for missing values!!!
        for e in cache['entities_unique']:
            ts = task["timepoints"][task['entities'] == e]
            v1 = X[task['entities'] == e, s1]
            ax.plot(np.arange(len(v1)), v1, color="grey", alpha=0.5)

        props = {
            'boxprops':{'facecolor':'none', 'edgecolor':'black'},
    #         'medianprops':{'color':'green'},
    #         'whiskerprops':{'color':'blue'},
    #         'capprops':{'color':'yellow'}
        }

        sns.boxplot(x=cache["timepoints_idx"], y=X[:,s1], **props, ax=ax)
        feature = task["feature_names"][s1]
        ax.set(title=feature, xlabel="timepoints", ylabel="feature value")
        
        plt.show()

In [ ]:
def calculate_basic_correlation_stats(task, cache, random_state=42):
    
    # shortcut for features
    X = task["features"]
    
    # calculate spearman correlation matrices
    if "n" not in cache:
        print("  * N per time point: collect")
        cache["n"] = {
            t:X[t == task["timepoints"]].shape[0]
            for t in cache["timepoints_unique"]}
    else:
        print("  * N per time point: cached")

    # calculate spearman correlation matrices
    if "correlation_matrices" not in cache:
        print("  * Correlation matrices: calculate")
        cache["correlation_matrices"] = {
            t:nedis.base.calculate_correlation_matrix(
                X[t == task["timepoints"]], 
                spearman=True) 
            for t in cache["timepoints_unique"]}
    else:
        print("  * Correlation matrices: cached")

    # calculate coordinates for each timepoint
    if "coordinates" not in cache:
        print("  * Coordinates: calculate")
        coordinates = {}
        for yy, correlation_matrix in cache["correlation_matrices"].items(): 
            tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state, init="pca", learning_rate="auto")
            coo = tsne.fit_transform(abs(correlation_matrix))
            coordinates[yy] = coo
        cache["coordinates"] = coordinates
    else:
        print("  * Coordinates: cached")

In [ ]:
def calculate_correlation_disruption(task, cache):
    
    # calculate disruptions
    if "disruption_matrices" not in cache:
        print("  * Disruption matrices: calculate")
        disruption_matrices = {
            t:nedis.base.calculate_correlation_disruption_matrix(
                task["features"], 
                X_ref=task["features"][task["timepoints"] == t], 
                correlation_function="spearman",
                mode="direction")
            for t in cache["timepoints_unique"]
        }
        cache["disruption_matrices"] = disruption_matrices
    else:
        print("  * Disruption matrices: cached")
        
    # calculate disruptions
    if "disruption_matrices_robust" not in cache:
        print("  * Disruption matrices (robust): calculate")
        disruption_matrices = {
            t:nedis.base.calculate_correlation_disruption_matrix_robust(
                task["features"], 
                idx_ref=task["timepoints"] == t, 
                correlation_function="spearman",
                mode="direction")
            for t in cache["timepoints_unique"]
        }
        cache["disruption_matrices_robust"] = disruption_matrices
    else:
        print("  * Disruption matrices (robust): cached")
      
    
    # calculate disruptions
    if "disruption_matrices_default" not in cache:
        print("  * Disruption matrices (default): calculate")
        disruption_matrices = {
            t:nedis.base.calculate_correlation_disruption_matrix(
                task["features"], 
                X_ref=task["features"][task["timepoints"] == t], 
                correlation_function="spearman",
                mode="default")
            for t in cache["timepoints_unique"]
        }
        cache["disruption_matrices_default"] = disruption_matrices
    else:
        print("  * Disruption matrices (default): cached")
    
    # calculate disruptions
    if "disruption_matrices_default_robust" not in cache:
        print("  * Disruption matrices (default, robust): calculate")
        disruption_matrices = {
            t:nedis.base.calculate_correlation_disruption_matrix_robust(
                task["features"], 
                idx_ref=task["timepoints"] == t, 
                correlation_function="spearman",
                mode="default")
            for t in cache["timepoints_unique"]
        }
        cache["disruption_matrices_default_robust"] = disruption_matrices
    else:
        print("  * Disruption matrices (default, robust): cached")

In [ ]:
def format_ax(ax):
    # https://stackoverflow.com/a/27361819/991496

    # Hide the right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    # Only show ticks on the left and bottom spines
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

In [ ]:
def format_label(s):
    s = re.sub("_", " ", s)
    s = re.sub("^[0-9]+ ", "", s)
    s = re.sub("((LPS)|(IL))100", r"\1", s)
    return s
format_label("999 test LPS100")

In [ ]:
def find_feature_pairs(task, cache, pvalue_mean_var_threshold=0.5):

    from matplotlib import gridspec
    from scipy.stats import norm
    def fisher_z(r): return np.arctanh(r)
    
    # correlation matrices across timepoints (axis=0)
    correlation_matrices_array = np.array([cache["correlation_matrices"][t] for t in cache["timepoints_unique"]])
    
    # mask to select correlations across timepoints for all feature pairs passing homoscedasticity threshold
    msk_homo = np.all(cache["pvalues_mean_var"] > pvalue_mean_var_threshold, axis=0)
    msk_homo_rows = np.tile(msk_homo, correlation_matrices_array.shape[-1])
    msk_homo_cols = np.repeat(msk_homo, correlation_matrices_array.shape[-1])
    msk_homo_2d = msk_homo_rows & msk_homo_cols
    print(sum(msk_homo_2d))

    # difference of correlations across time
    correlation_matrices_array_min = np.min(correlation_matrices_array, axis=0)
    correlation_matrices_array_max = np.max(correlation_matrices_array, axis=0)
    diff = np.abs(correlation_matrices_array_max - correlation_matrices_array_min).flatten()
    
    # find feature paris woth top correlation differences
    idx_diff_homo = np.arange(diff.size)[msk_homo_2d]
    values_diff = diff[idx_diff_homo]
    values_diff_order = np.argsort(-values_diff)
    idx_top_k_rows, idx_top_k_cols = np.unravel_index(idx_diff_homo[values_diff_order], correlation_matrices_array.shape[1:])

    X = task['features']
    for i, (s1, s2, d) in enumerate(zip(idx_top_k_rows, idx_top_k_cols, values_diff[values_diff_order])):
        
        print(s1, s2, d, cache["pvalues_mean_var"][:,s1], cache["pvalues_mean_var"][:,s2])
        
        fig, axes = plt.subplots(1, 4, figsize=(3 * 3.2,3), dpi=300)
        from matplotlib import gridspec
        gs = gridspec.GridSpec(1, 6)


#         # TODO: does no work for missing values!!!
#         for e in cache['entities_unique']:
#             ts = task["timepoints"][task['entities'] == e]
#             v1 = X[task['entities'] == e, s1]
#             ax.plot(np.arange(len(v1)), v1, color="grey", alpha=0.5)

        # palette = "viridis"
        palette = "plasma"
        props = {
            # 'boxprops':{'facecolor':'none', 'edgecolor':'black'},
    #         'medianprops':{'color':'green'},
    #         'whiskerprops':{'color':'blue'},
    #         'capprops':{'color':'yellow'}, 
            "palette": palette
        }
            
        def format_label(s):
            s = re.sub("_", " ", s)
            s = re.sub("^[0-9]+ ", "", s)
            s = re.sub("((LPS)|(IL))100", r"\1", s)
            return s
    
        print(f"feature pair: {i}")
    
        ax = plt.subplot(gs[0, 0:1])
        sns.boxplot(x=task["timepoints"], y=X[:,s1], **props, ax=ax)
        feature1 = task["feature_names"][s1]
        ax.set(
            xlabel="timepoint", 
            ylabel=format_label(feature1),
            xticklabels=[timepoint_map[t.get_text()] for t in ax.get_xticklabels()]
        )
        format_ax(ax)
        
        ax = plt.subplot(gs[0, 1:2])
        sns.boxplot(x=task["timepoints"], y=X[:,s2], **props, ax=ax)
        feature2 = task["feature_names"][s2]
        ax.set(
            xlabel="timepoint", 
            ylabel=format_label(feature2),
            xticklabels=[timepoint_map[t.get_text()] for t in ax.get_xticklabels()])
        format_ax(ax)
        
        ax = plt.subplot(gs[0, 2:4])
        X1 = X.copy()
        colors = sns.color_palette(palette, n_colors=4)
        for i_t, t in enumerate(cache["timepoints_unique"]):
            msk = task["timepoints"] == t
            # msk &= X1[:,s1] < 0.05
            X1[msk] = sklearn.preprocessing.MinMaxScaler().fit_transform(X1[msk])
            sns.regplot(
                x=X1[msk,s1],
                y=X1[msk,s2], 
                marker="x", 
                ax=ax, 
                line_kws=dict(linestyle="--"), 
                scatter_kws=dict(s=20),
                color=colors[i_t])
        ax.set(
            xlabel=format_label(feature1), 
            ylabel=format_label(feature2))
        format_ax(ax)
        
        ax = plt.subplot(gs[0, 4:6])
        ax.plot(
            cache["timepoints_unique"], 
            [cache["correlation_matrices"][t][s1,s2] for t in cache["timepoints_unique"]], color="grey", linestyle="--")
        ax.scatter(
            cache["timepoints_unique"], 
            [cache["correlation_matrices"][t][s1,s2] for t in cache["timepoints_unique"]], c=colors, s=100, zorder=100)
        correlation_values = [cache["correlation_matrices"][t][s1,s2] for t in cache["timepoints_unique"]]
        for t1, c1 in zip(cache["timepoints_unique"], correlation_values):
            for t2, c2 in zip(cache["timepoints_unique"], correlation_values):
                # calculate whether c1 and c2 are significantly different
                n1 = cache["n"][t1]
                n2 = cache["n"][t2]
                z = (fisher_z(c1) - fisher_z(c2)) / np.sqrt(1/(n1-3) + 1/(n2-3))
                p = 2*norm.sf(abs(z))
                print(t1, t2, n1, n2, c1, c2, p)
        
        
        fig.canvas.draw()
        ax.set(
            xlabel="timepoint", 
            ylabel="correlation",
            xticklabels=[timepoint_map[t.get_text()] if t.get_text() in timepoint_map else "XXX" for t in ax.get_xticklabels()]
        )
        format_ax(ax)
        
        fig.tight_layout() 
        fig.savefig(output_dir / f"feature-pair_{i}.png", bbox_inches="tight")
        fig.savefig(output_dir / f"feature-pair_{i}.pdf", bbox_inches="tight")
        
        plt.show()
        
        if i > 10:
            break

# Figure 1

In [ ]:
task = select_task("multiomics-immune-system-ref-all-all")

In [ ]:
timepoint_map = {
    "1": "1", "2": "2", "3": "3", "4": "PP"
}

## There are variables that do not change mean and variance but correlation

In [ ]:
cache = {}

In [ ]:
prepare_variables(task, cache)

In [ ]:
check_homoescedasticity(task, cache, plot_n_features=0)

In [ ]:
calculate_basic_correlation_stats(task, cache)

In [ ]:
def fisher_z(r): return np.arctanh(r)

c1 = 0.6862745098039216
c2 = 0.4975490196078432



In [ ]:
find_feature_pairs(task, cache, pvalue_mean_var_threshold=0.5)

In [ ]:
# f1 = np.argwhere(task["feature_names"] == "219_intMCs_ERK_LPS100").squeeze()

## Interesting modules

In [ ]:
from nedis.experiments import load_component
from nedis.visualization import nx_plot

In [ ]:
exp_name = "regression___params___transformer_default___task_multiomics-immune-system-pp-first-ref-all-all___handle-heteroscedasticity_robust___groups_True___random-state_120"
exp_dir = pathlib.Path("../_out/realworld") / exp_name

In [ ]:
transformer = load_component("transformer", output_dir=exp_dir)

In [ ]:
feature_names = load_component("feature_names", output_dir=exp_dir)

In [ ]:
y = load_component("y", output_dir=exp_dir)

In [ ]:
entities = load_component("entities", output_dir=exp_dir)

In [ ]:
coordinates_dict = load_component("coordinates_dict", output_dir=exp_dir)
coo = coordinates_dict[0]

In [ ]:
correlation_matrices_dict = load_component("correlation_matrices_dict", output_dir=exp_dir)
correlation_matrix = correlation_matrices_dict[3]
correlation_threshold = -1

In [ ]:
disruption_matrices_dict = load_component("disruption_matrices_dict", output_dir=exp_dir)

In [ ]:
clusters = sorted(transformer.clusters_, key=lambda c: -np.abs(c["reference_score"]))
cluster = clusters[0]

In [ ]:
edges = [
    (src, dst) for src, dst in zip(*cluster["edges"].nonzero())
    if src != dst and 
        abs(correlation_matrix[src, dst]) >= correlation_threshold]
rows = list({src for src, _ in edges})
columns = list({dst for _, dst in edges})

In [ ]:
nodes_pos = {
    i:p 
    for i, p in enumerate(coo) 
    if i in cluster["rows"] or i in cluster["columns"]}

In [ ]:
# if edges_mode == "correlation":
#     def edges_width(g, src, dst, d):
#         abscor = abs(correlation_matrix[src, dst])
#         if src == dst and not np.isclose(correlation_matrix[src,dst], 1):
#             print(correlation_matrix[src,dst])
#         t = 0.1
#         if abscor < t:
#             return 0
#         else:
#             return (abscor - t) / (1 - t) * 5

#     def edges_edge_color(g, src, dst, d):
#         cmap = sns.color_palette(palette="vlag", as_cmap=True)
#         cor = correlation_matrix[src, dst]
#         c = (cor + 1) / 2
#         color = cmap(c)
#         color = (*color[:-1], 0.7)
#         return color


In [ ]:
disruption_matrices = disruption_matrices_dict[cluster["reference_label"]]
dis = np.max([np.max(np.abs(np.median(d, axis=0))) for d in disruption_matrices_dict.values()])
norm = plt.Normalize(-dis, dis)

In [ ]:
cluster1 = [
"298_intMCs_CREB_Unstim",
"299_M-MDSC_CREB_Unstim",
# "374_M-MDSC_MAPKAPK2_Unstim",
"296_cMCs_CREB_Unstim",
"302_pDCs_CREB_Unstim"
]

In [ ]:
cluster2  = [
"494_CD8+Tcells_naive_STAT3_Unstim",
"486_CD4+Tcells_mem_STAT3_Unstim",
"488_CD4+Tcells_STAT3_Unstim",
"490_CD45RA-Tregs_STAT3_Unstim",
"508_Tregs_STAT3_Unstim",
"487_CD4+Tcells_naive_STAT3_Unstim",
"489_CD45RA+Tregs_STAT3_Unstim",
"496_cMCs_STAT3_Unstim",
"491_CD56+CD16-NKcells_STAT3_Unstim",
"507_TCRgd+Tcells_STAT3_Unstim",
"505_Tbet+CD8+Tcells_mem_STAT3_Unstim",
"493_CD8+Tcells_mem_STAT3_Unstim",
"503_Tbet+CD4+Tcells_mem_STAT3_Unstim",
"414_CD45RA+Tregs_p38_Unstim",
"422_Gr_p38_Unstim",
"478_Tbet+CD4+Tcells_mem_STAT1_Unstim",
"415_CD45RA-Tregs_p38_Unstim",
"433_Tregs_p38_Unstim",
"419_CD8+Tcells_naive_p38_Unstim",
"412_CD4+Tcells_naive_p38_Unstim",
"413_CD4+Tcells_p38_Unstim",
"410_CD16+CD56-NKcells_p38_Unstim",
"428_Tbet+CD4+Tcells_mem_p38_Unstim",
"468_CD8+Tcells_mem_STAT1_Unstim",
"480_Tbet+CD8+Tcells_mem_STAT1_Unstim",
"461_CD4+Tcells_mem_STAT1_Unstim",
"482_TCRgd+Tcells_STAT1_Unstim",
# "287_CD4+Tcells_naive_CREB_Unstim",
# "294_CD8+Tcells_naive_CREB_Unstim",
#
"465_CD45RA-Tregs_STAT1_Unstim",
"483_Tregs_STAT1_Unstim",
# "290_CD45RA-Tregs_CREB_Unstim",
]

In [ ]:
import nedis.visualization
points = coo[[f in cluster1 for f in feature_names]]
hull1 = nedis.visualization.calculate_hull(points, scale=1.2, interpolation="cubic")

In [ ]:
import nedis.visualization
points = coo[[f in cluster2 for f in feature_names]]
hull2 = nedis.visualization.calculate_hull(points, scale=1.1, interpolation="cubic")

In [ ]:
coordinates_dict.keys()

In [ ]:
timepoint_map_pp = {
    0: "Postpartum", 1: "Trimester 1", 2: "Trimester 2", 3: "Trimester 3\n(reference)"
} 

In [ ]:
color1 = "#56941E" # green
color2 = "purple"  # purple
color_neutral = "black"

scale = 2
fig, axes = plt.subplots(1, 1, figsize=(3 * scale, 3 * scale), squeeze=False)
# for i, (key, _) in enumerate(coordinates_dict.items()):
for i, key in enumerate([3]):
    ax = axes[0, i]
    ax.axis("off")
    ax.scatter(coo[:,0], coo[:,1], color="lightgrey")
    
    def nodes_node_size(g, n, d):
        if n in rows and n in columns:
            return 100
        # elif n in rows:
        #     return 40
        # elif n in columns:
        #     return 40
        else:
            return 0.1

    def nodes_node_color(g, n, d):
        if feature_names[n] in cluster1:
            return color1
        if feature_names[n] in cluster2:
            return color2
        if n in rows and n in columns:
            return color_neutral
        # elif n in rows:
        #     return "red"
        # elif n in columns:
        #     return "blue"
        else:
            return "grey"

    correlation_matrix = correlation_matrices_dict[i]
    
    def edges_width(g, src, dst, d):
        abscor = abs(correlation_matrix[src, dst])
        if src == dst and not np.isclose(correlation_matrix[src,dst], 1):
            print(correlation_matrix[src,dst])
        t = 0.1
        if abscor < t:
            return 0
        else:
            return (abscor - t) / (1 - t) * 5

    def edges_edge_color(g, src, dst, d):
        cmap = sns.color_palette(palette="vlag", as_cmap=True)
        cor = correlation_matrix[src, dst]
        c = (cor + 1) / 2
        color = cmap(c)
        color = (*color[:-1], 0.7)
        return color

    nx_plot(
        # nodes=np.arange(coo.shape[0]),
        nodes_pos=nodes_pos,
        nodes_args=dict(
            node_size=nodes_node_size,
            node_color=nodes_node_color,
            edgecolors="white"
        ),
        edges=edges,
        edges_args=dict(
            width=edges_width,
            edge_color=edges_edge_color,
        ),
        # nodes_labels=labels,
        nodes_labels_args={
            "font_size": 2
        },
        ax=ax)
    
    ax.plot(hull1[:,0], hull1[:,1], color=color1, linewidth=2)
    ax.plot(hull2[:,0], hull2[:,1], color=color2, linewidth=2)
    
    
    ax.annotate(
        "Innate immune\npopulations\npCREB\nlevels", hull1[np.argmax(hull1[:,0])], 
        xytext=(60, 30), textcoords="offset points",
        va="center", ha="right",
        fontweight="bold",
        fontsize=14,
        color=color1)
    ax.annotate(
        "CD4 and CD8 T cells\nSTAT1, STAT3,\npP38 levels", hull2[np.argmax(hull2[:,0])], 
        xytext=(100, 20), textcoords="offset points",
        va="center", ha="right",
        fontweight="bold",
        fontsize=14,
        color=color2)
    
    ax.annotate(
        timepoint_map_pp[3], 
        (0.1, 0.37), va="top", ha="left", color="black", fontsize=20, xycoords="axes fraction")
    
fig.savefig(output_dir / f"networks_correlation.png", bbox_inches="tight")
fig.savefig(output_dir / f"networks_correlation.pdf", bbox_inches="tight")

In [ ]:
scale = 2
fig, axes = plt.subplots(1, 4, figsize=(4 * 4 * scale, 4 * scale), squeeze=False)
# for i, (key, _) in enumerate(coordinates_dict.items()):
for i, key in enumerate([1,2,3,0]):
    ax = axes[0, i]
    ax.axis("off")
    ax.scatter(coo[:,0], coo[:,1], color="lightgrey")
    
    def nodes_node_size(g, n, d):
        if n in rows and n in columns:
            return 100
        # elif n in rows:
        #     return 40
        # elif n in columns:
        #     return 40
        else:
            return 0.1

    def nodes_node_color(g, n, d):
        if feature_names[n] in cluster1:
            return color1
        if feature_names[n] in cluster2:
            return color2
        if n in rows and n in columns:
            return "black"
        # elif n in rows:
        #     return "red"
        # elif n in columns:
        #     return "blue"
        else:
            return "grey"

    def edges_width(g, src, dst, d):
        dis = np.median(disruption_matrices[y == key, src, dst])
        if True:\
        # if \
        #         (feature_names[src] in cluster1 \
        #             and feature_names[dst] in cluster2) \
        #         or (feature_names[src] in cluster2 \
        #             and feature_names[dst] in cluster1):
            return np.abs((norm(dis) - 0.5)) * 2 * 10
        else:
            return .5

    def edges_edge_color(g, src, dst, d):
        cmap = sns.color_palette(palette="vlag", as_cmap=True)
        dis = np.median(disruption_matrices[y == key, src, dst])
        color = cmap(norm(dis))
        color = (*color[:-1], np.abs((norm(dis) - 0.5)) * 2)
        
        # if True:\
        if \
            (feature_names[src] in cluster1 \
                and feature_names[dst] in cluster2) \
            or (feature_names[src] in cluster2 \
                and feature_names[dst] in cluster1):
            return color
        else:
            return [0.7,0.7,0.7,0.05]   

    nx_plot(
        # nodes=np.arange(coo.shape[0]),
        nodes_pos=nodes_pos,
        nodes_args=dict(
            node_size=nodes_node_size,
            node_color=nodes_node_color,
            edgecolors="white"
        ),
        edges=edges,
        edges_args=dict(
            width=edges_width,
            edge_color=edges_edge_color,
        ),
        # nodes_labels=labels,
        nodes_labels_args={
            "font_size": 2
        },
        ax=ax)
    
    ax.plot(hull1[:,0], hull1[:,1], color=color1, linewidth=2)
    ax.plot(hull2[:,0], hull2[:,1], color=color2, linewidth=2)
    
    ax.annotate(
        timepoint_map_pp[key], 
        (0.1, 0.37), va="top", ha="left", color="black", fontsize=20, xycoords="axes fraction")
    
fig.savefig(output_dir / f"networks_disruption.png", bbox_inches="tight")
fig.savefig(output_dir / f"networks_disruption.pdf", bbox_inches="tight")

In [ ]:
dis = disruption_matrices_dict[3]
dis = np.mean(dis[:,cluster["edges"].toarray() == 1], axis=(1))

fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=300)

yy = y.copy()
yy[yy == 0] = 4
grouped_spline_plot(
    x=yy, 
    y=dis, 
    color=sns.color_palette()[0],
    # groups=entities,
    groups_kwargs=dict(color="grey", linewidth=1, alpha=0.2))

format_ax(ax)
ax.set(xlabel="timepoint", ylabel="disruption", xticklabels=["1", "2", "3", "PP"])

fig.savefig(output_dir / f"disruption_cluster.png", bbox_inches="tight")
fig.savefig(output_dir / f"disruption_cluster.pdf", bbox_inches="tight")

In [ ]:
msk = np.array([f in cluster1 or f in cluster2 for f in feature_names])
dis = disruption_matrices_dict[3]
dis = dis[:,:,msk][:,msk,:]
dis = np.mean(dis, axis=(1,2))

fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)

yy = y.copy()
yy[yy == 0] = 4
grouped_spline_plot(
    yy, 
    dis, 
    color=sns.color_palette()[0], 
    # groups=entities,
    groups_kwargs=dict(color="grey", linewidth=1, alpha=0.2))

format_ax(ax)
ax.set(xlabel="timepoint", ylabel="disruption", xticklabels=["1", "2", "3", "PP"])

fig.savefig(output_dir / f"disruption_cluster_subset_full.png", bbox_inches="tight")
fig.savefig(output_dir / f"disruption_cluster_subset_full.pdf", bbox_inches="tight")

In [ ]:
msk1 = np.array([f in cluster1 for f in feature_names])
msk2 = np.array([f in cluster2 for f in feature_names])
dis = disruption_matrices_dict[3]
dis = dis[:,:,msk1][:,msk2,:]
dis = np.mean(dis, axis=(1,2))

fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)

yy = y.copy()
yy[yy == 0] = 4
grouped_spline_plot(
    yy, 
    dis, 
    color=sns.color_palette()[0], 
    # groups=entities,
    groups_kwargs=dict(color="grey", linewidth=1, alpha=0.2))

format_ax(ax)
ax.set(xlabel="timepoint", ylabel="disruption", xticklabels=["1", "2", "3", "PP"])

fig.savefig(output_dir / f"disruption_cluster_subset.png", bbox_inches="tight")
fig.savefig(output_dir / f"disruption_cluster_subset.pdf", bbox_inches="tight")